# 03 — Vision 캡션 프롬프트 EDA

spec: `docs/superpowers/specs/2026-06-03-vision-caption-eda-design.md`
plan: `docs/superpowers/plans/2026-06-03-vision-caption-eda.md`

입력: 02 핸드오프(`vision_manifest.parquet` 1,737 + `thumbs/`). **전부 로컬 Ollama, 외부 전송 0(ADR-0001).**
처리량 실측: gemma4:e2b ~8.3s/장, gemma4:26b ~18.6s/장. 무거운 캡션 생성은 캐시(parquet)로 재실행 저렴.

In [1]:
import json, time, random
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter
import pillow_heif; pillow_heif.register_heif_opener()
import ollama

SEED = 42
random.seed(SEED); np.random.seed(SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CACHE = ROOT / "data" / "eda_cache"
MANIFEST = CACHE / "vision_manifest.parquet"
THUMB_DIR = CACHE / "thumbs"

CAP_MODELS = ["gemma4:e2b", "gemma4:26b"]
EMB_MODEL = "qwen3-embedding:8b"
print("ROOT:", ROOT)

ROOT: /Users/shingh/works/eddr


In [2]:
# §0 Ollama 연결·모델 존재 확인
installed = {m.model for m in ollama.list().models}
need = set(CAP_MODELS) | {EMB_MODEL}
missing = need - installed
assert not missing, f"누락 모델: {missing} — ollama pull 필요"
print("Ollama OK:", sorted(need))

Ollama OK: ['gemma4:26b', 'gemma4:e2b', 'qwen3-embedding:8b']


In [ ]:
# §0 manifest 로드·sanity
import unicodedata as ud
m = pd.read_parquet(MANIFEST)
# 02가 google_takeout(3rd 소스)을 manifest에 합류(3,122행, source 컬럼 신설) → 03은 프롬프트 선택 EDA라
# local 평가 모집단만 사용(takeout 캡션은 빌드 ⑤ 실제 비전 단계에서). 구 manifest(source 컬럼 없음)도 호환.
if "source" in m.columns:
    m = m[m["source"] == "local"].reset_index(drop=True)
assert len(m) == 1737, len(m)
assert m["thumb_path"].notna().all()
m["ext"] = m["local_path"].str.lower().str.rsplit(".", n=1).str[-1]
m["folder_top"] = m["folder_top"].map(lambda s: ud.normalize("NFC", s))  # macOS NFD -> NFC (한글 매칭)
print(len(m), "행 · bucket:", m["bucket"].value_counts().to_dict())

In [4]:
# §1 픽셀 지표 순수 함수
def pixel_metrics(thumb_path: str) -> dict:
    """썸네일 저비용 지표: 밝기·노출클리핑·흐림·종횡비."""
    im = Image.open(thumb_path).convert("L")
    a = np.asarray(im, dtype=np.float32)
    lap = np.asarray(im.filter(ImageFilter.FIND_EDGES), dtype=np.float32)
    h, w = a.shape
    return {"brightness": float(a.mean()), "clip_lo": float((a < 8).mean()),
            "clip_hi": float((a > 247).mean()), "blur": float(lap.var()), "aspect": float(w / h)}

In [5]:
# §1 합성 이미지 assert
def _synth(val):
    p = CACHE / f"_synth_{val}.jpg"; Image.new("L", (64, 48), val).save(p); return str(p)
mb = pixel_metrics(_synth(0)); mw = pixel_metrics(_synth(255))
assert mb["brightness"] < 1 and mw["brightness"] > 254
assert mb["clip_lo"] == 1.0 and mw["clip_hi"] == 1.0
assert abs(mb["aspect"] - 64/48) < 1e-6
print("pixel_metrics assert OK")
for v in (0, 255): (CACHE / f"_synth_{v}.jpg").unlink()

pixel_metrics assert OK


In [6]:
# §1 전수 픽셀 지표 + 캐시
PIX = CACHE / "pixel_metrics_03.parquet"
if PIX.exists():
    px = pd.read_parquet(PIX); print("픽셀지표 캐시 로드:", len(px))
else:
    rows, fails = [], []
    t0 = time.time()
    for tp in m["thumb_path"]:
        try: rows.append({"thumb_path": tp, **pixel_metrics(tp)})
        except Exception as e: fails.append((tp, str(e)))
    px = pd.DataFrame(rows); px.to_parquet(PIX)
    print(f"{len(px)} 지표 · 실패 {len(fails)} · {time.time()-t0:.0f}s")
mx = m.merge(px, on="thumb_path", how="left")
assert mx["brightness"].notna().mean() > 0.99
print("merge OK", len(mx))

픽셀지표 캐시 로드: 1737
merge OK 1737


In [7]:
# §1 manifest 재집계 + png/jpg 화질 비교
print("== folder_top top10 =="); print(m["folder_top"].value_counts().head(10))
print("== 포맷 ==", m["ext"].value_counts().to_dict())
print("== match_confidence ==", m["match_confidence"].value_counts().to_dict())
print("== 해상도 MP 분위 =="); print((mx.width*mx.height/1e6).describe(percentiles=[.1,.5,.9]).round(2))
print("== png vs jpg 밝기/흐림 중앙값 ==")
print(mx.groupby("ext")[["brightness","blur","clip_lo","clip_hi"]].median().round(2))

== folder_top top10 ==
folder_top
2019_이탈리아       583
2022_아이슬란드      236
2018mongolia    152
하율              113
wedding          85
반디               84
기타               80
마음의 빚 우토로        64
utoro            58
200620_23_제주     52
Name: count, dtype: int64
== 포맷 == {'jpg': 1143, 'png': 594}
== match_confidence == {'none': 1310, 'medium': 400, 'high': 27}
== 해상도 MP 분위 ==
count    1737.00
mean        8.68
std        10.11
min         0.38
10%         0.85
50%         1.71
90%        25.96
max        48.84
dtype: float64
== png vs jpg 밝기/흐림 중앙값 ==
     brightness     blur  clip_lo  clip_hi
ext                                       
jpg      118.60  1146.02      0.0      0.0
png      116.93  1594.33      0.0      0.0


In [8]:
# §1 어려운 축 식별 (→ §2 샘플 층화)
mx["hard"] = (mx["brightness"] < 40) | (mx["clip_lo"] > 0.5) | (mx["blur"] < mx["blur"].quantile(0.1))
print("어려운 후보(저조도/암부클리핑/흐림 하위10%):", int(mx["hard"].sum()))

어려운 후보(저조도/암부클리핑/흐림 하위10%): 299


## §2 정성 캡션 비교 (프롬프트 3종 × 모델 2)

24장 층화 샘플에 3프롬프트 × 2모델 = 144 캡션. 썸네일+6캡션 표를 보고 best 조합을 고른다.

In [9]:
# §2 프롬프트 3종 (영어 출력 고정)
PROMPTS = {
  "P1_concise": "Describe this photo in one concise English sentence.",
  "P2_struct": ("Describe this photo in English. Cover: place/setting type, "
      "main objects or subjects, activity or event, any visible text, overall mood. "
      "Be specific and factual; do not guess the names of people."),
  "P3_hybrid": ("Describe this photo in English in 1-2 sentences, then list 5-10 "
      "searchable keywords (nouns: places, objects, events, food, activities).\n"
      "Format:\nCaption: <sentence>\nKeywords: <comma-separated>"),
}
print({k: len(v) for k, v in PROMPTS.items()})

{'P1_concise': 52, 'P2_struct': 194, 'P3_hybrid': 191}


In [10]:
# §2 층화 샘플 24장
def stratified_sample(df, n=24, seed=SEED):
    df = df.copy()
    df["bright_band"] = pd.cut(df["brightness"], [0,60,120,256], labels=["dark","mid","bright"])
    picks = df.groupby(["bucket","ext","bright_band"], observed=True).sample(1, random_state=seed)
    extra = df[df["hard"]].sample(min(4, int(df["hard"].sum())), random_state=seed)
    out = pd.concat([picks, extra]).drop_duplicates("thumb_path")
    if len(out) < n:
        rest = df[~df["thumb_path"].isin(out["thumb_path"])].sample(n-len(out), random_state=seed)
        out = pd.concat([out, rest])
    return out.head(n).reset_index(drop=True)

samp = stratified_sample(mx)
assert len(samp) == 24
assert samp["folder_top"].nunique() >= 6 and samp["ext"].nunique() >= 2
print("샘플 24 OK · folders:", samp["folder_top"].nunique(), "· png:", int((samp.ext=="png").sum()))

샘플 24 OK · folders: 11 · png: 10


In [11]:
# §2 Ollama 캡션 함수 (flag-skip)
def caption(thumb_path, model, prompt):
    try:
        r = ollama.chat(model=model, messages=[
            {"role":"user","content":prompt,"images":[thumb_path]}], options={"seed":SEED})
        return r["message"]["content"].strip()
    except Exception as e:
        return f"__ERR__:{e}"

In [12]:
# §2 144 캡션 생성 + 캐시 (무거움 ~32분; 에이전트가 background로 미리 생성)
CAPS = CACHE / "captions_03.parquet"
if CAPS.exists():
    caps = pd.read_parquet(CAPS); print("캡션 캐시 로드:", len(caps))
else:
    recs, t0 = [], time.time()
    for _, row in samp.iterrows():
        for model in CAP_MODELS:
            for pid, ptext in PROMPTS.items():
                recs.append({"thumb_path":row.thumb_path,"folder_top":row.folder_top,
                    "model":model,"prompt_id":pid,"caption":caption(row.thumb_path,model,ptext)})
    caps = pd.DataFrame(recs); caps.to_parquet(CAPS)
    print(f"{len(caps)} 캡션 · {time.time()-t0:.0f}s · 에러 {int(caps.caption.str.startswith('__ERR__').sum())}")
assert len(caps) == 24*2*3

캡션 캐시 로드: 144


In [13]:
# §2 정성 비교 표 (썸네일 + 6캡션)
from IPython.display import display, HTML
def show_row(thumb):
    sub = caps[caps.thumb_path==thumb]
    html = f'<img src="{thumb}" width="240"><table>'
    for model in CAP_MODELS:
        for pid in PROMPTS:
            cell = sub[(sub.model==model)&(sub.prompt_id==pid)]
            c = cell.caption.iloc[0] if len(cell) else "(없음)"
            html += f"<tr><td><b>{model}/{pid}</b></td><td>{c}</td></tr>"
    display(HTML(html+"</table>"))
for thumb in samp.thumb_path: show_row(thumb)

gemma4:e2b/P1_concise,"Dramatic, diffused light streams through heavy, dark clouds over a blurred landscape."
gemma4:e2b/P2_struct,"This is a photograph characterized by heavy darkness and atmospheric effects. **Place/Setting Type:** Outdoor landscape, likely an open field or plain. **Main Objects or Subjects:** The image is dominated by large, dark silhouettes in the foreground and middle ground. There is a faint, hazy horizon line with a dark silhouette of vegetation or trees visible in the distance. **Activity or Event:** The scene captures a landscape viewed under dramatic, diffused lighting, suggesting an overcast sky or a specific atmospheric event. The foreground elements are heavily blurred and darkened. **Visible Text:** There is no visible text in the image. **Overall Mood:** The mood is dark, dramatic, and hazy. The use of deep shadows and diffused light creates a somber or atmospheric feeling."
gemma4:e2b/P3_hybrid,"Caption: A moody, blurred landscape is visible beneath dark, silhouetted foreground elements, emphasizing diffused light and an overcast atmosphere. Keywords: landscape, sky, clouds, horizon, mist, outdoor, dark, nature, light"
gemma4:26b/P1_concise,"A blurry, dark, and atmospheric photograph of a misty, rainy landscape."
gemma4:26b/P2_struct,"This is a blurry, out-of-focus photograph of a rural landscape during a storm. The shot appears to be taken from inside a vehicle, as large, dark, indistinct shapes—likely part of a car's interior or window frame—occupy the left and upper foreground. The setting features an open field or meadow in the midground, meeting a dark, heavy horizon. The sky is filled with thick, dark gray and blue clouds, with visible vertical streaks of light suggesting falling rain or rain shafts. On the distant horizon to the right, a small, bright white object, which appears to be a vehicle, is visible. There is no visible text in the image. The overall mood is gloomy, dramatic, and atmospheric."
gemma4:26b/P3_hybrid,"Caption: A blurry and dark photograph shows a landscape under a heavy, stormy sky with visible rain. Keywords: sky, clouds, rain, landscape, storm, field, nature, weather, gloom"


gemma4:e2b/P1_concise,"A young woman sits outdoors at a cafe, looking down at a book or menu while enjoying the atmosphere."
gemma4:e2b/P2_struct,"This is an outdoor photograph capturing a relaxed social scene in what appears to be a cafe or restaurant setting. **Place/Setting Type:** Outdoor seating area, likely a cafe or restaurant, characterized by architectural details like arches visible in the background. **Main Objects or Subjects:** The primary subject is a young woman in the foreground, looking down and reading an open book or magazine. Other people are partially visible in the background, seated at tables. **Activity or Event:** The subjects are engaged in leisure time, sitting and reading in an outdoor environment. **Visible Text:** No specific, legible text is visible in the image. **Overall Mood:** The mood is relaxed, casual, and leisurely, suggesting a peaceful daytime social experience."
gemma4:e2b/P3_hybrid,"Caption: A young woman is sitting at an outdoor cafe, engrossed in reading a book or menu while enjoying the warm atmosphere. Keywords: woman, cafe, outdoor dining, book, reading, restaurant, street, architecture, people, leisure"
gemma4:26b/P1_concise,"A young woman in a white top sits at an outdoor cafe, looking down at a menu."
gemma4:26b/P2_struct,"This photograph captures a scene at an outdoor cafe or restaurant terrace during the daytime. The central subject is a young woman with long, straight, light brown hair, wearing a white spaghetti-strap top and a delicate necklace. She is looking downward, focused on reading a menu. Surrounding her are other patrons, including a person in a pink patterned top to the left and another person in a blue and white patterned top to the right, both with their backs to the camera. In the blurred background, figures can be seen walking along a street lined with buildings, suggesting a public urban setting. In the bottom foreground, the top of a red and white aluminum can is partially visible. No legible text is present in the image. The overall mood of the photo is calm, quiet, and contemplative."
gemma4:26b/P3_hybrid,"Caption: A young woman with long brown hair sits at an outdoor cafe, focused on reading a menu. Keywords: woman, cafe, menu, outdoor dining, restaurant, lifestyle, summer, leisure"


gemma4:e2b/P1_concise,"A vast, tranquil landscape features a wide expanse of water, grassy foreground, and distant mountain ranges under a pale, cloudy sky."
gemma4:e2b/P2_struct,"**Place/Setting Type:** A vast, open, and flat landscape, likely a plain, steppe, or tundra environment, featuring a body of water and distant mountain ranges. **Main Objects or Subjects:** The image primarily features a wide expanse of water in the foreground, dark, grassy terrain, a distant, low mountain range, and a pale, cloudy sky. **Activity or Event:** The image depicts a static landscape scene, capturing a natural view of the horizon and the sky. There is no visible human activity. **Visible Text:** There is no visible text in the image. **Overall Mood:** The mood is expansive, quiet, and subdued. The soft, diffused light and muted colors create a peaceful and stark atmosphere."
gemma4:e2b/P3_hybrid,"Caption: A vast, open landscape features a calm body of water and rolling mountains under a soft, expansive sky. Keywords: Plains, mountains, river, landscape, sky, clouds, nature, horizon"
gemma4:26b/P1_concise,"A serene landscape of a calm river flowing through a grassy plain toward distant mountains under a soft, cloudy sky."
gemma4:26b/P2_struct,"This image captures a vast, expansive landscape featuring a wide body of water, grassy plains, and a distant mountain range under a large sky. * **Place/setting type:** A serene, open natural landscape, likely a coastal or river delta region near mountains. * **Main objects or subjects:** In the foreground, there is a dark, grassy field. The middle ground is occupied by a wide, calm body of water, such as a river estuary or fjord, with visible dark silt or sandbars. The background features a long, dark, flat-topped mountain ridge stretching across the horizon, which gives way to more rugged, snow-dusted peaks on the far right. The sky is a light blue, filled with scattered, thin, horizontal clouds. * **Activity or event:** There is no visible activity or event; it is a static landscape scene. * **Visible text:** There is no visible text in the image. * **Overall mood:** The mood is peaceful, tranquil, and majestic."
gemma4:26b/P3_hybrid,"Caption: A serene landscape featuring a wide, calm river flowing through a grassy plain towards distant, snow-capped mountains under a soft, cloudy sky. Keywords: landscape, river, mountains, clouds, nature, sky, glacier, water"


gemma4:e2b/P1_concise,"Dappled sunlight filters through a dense, shadowy forest, illuminating the foliage and scattering yellow specks across the scene."
gemma4:e2b/P2_struct,"**Place/Setting Type:** A dense, wooded area or forest. **Main Objects or Subjects:** Numerous tree trunks, branches, dense green foliage, and small yellow flying insects (likely butterflies or moths). **Activity or Event:** Dappled sunlight filters through the canopy, illuminating the forest floor and creating strong contrast between bright patches and deep shadows. The presence of the yellow specks suggests natural wildlife activity. **Visible Text:** There is no visible text in the image. **Overall Mood:** The mood is deep, shadowed, and tranquil, emphasizing the mysterious and lush nature of the natural environment."
gemma4:e2b/P3_hybrid,"Caption: Sunlight filters through the dense canopy of a lush green forest, illuminating the trees and foliage with dappled light. Keywords: forest, trees, sunlight, nature, greenery, woods, nature scene, foliage, light, outdoors"
gemma4:26b/P1_concise,"Fireflies twinkle amidst the dark, dense trees of a lush forest."
gemma4:26b/P2_struct,"This photograph depicts a dark, wooded forest setting, likely at night or during twilight. The main subjects include the dark, silhouetted trunks and branches of numerous trees, dense green foliage in both the foreground and background, and many small, glowing yellow points of light, which appear to be fireflies, scattered throughout the scene. The activity captured is the natural phenomenon of fireflies fluttering among the trees and low-lying vegetation. There is no visible text in the image. The overall mood is magical, serene, and tranquil."
gemma4:26b/P3_hybrid,"Caption: A magical night scene featuring numerous glowing fireflies dancing among the trees in a dark, dense forest. Keywords: forest, fireflies, night, trees, nature, woodland, bioluminescence, summer, darkness, lights"


gemma4:e2b/P1_concise,A view framed by a rustic wooden structure overlooks a dense residential area featuring foliage and yellow machinery.
gemma4:e2b/P2_struct,"This is a photograph depicting a view from within a structure, framed by rough wooden beams and framework. **Place/Setting Type:** The setting is an urban or suburban residential area characterized by densely packed housing. **Main Objects or Subjects:** The dominant subjects are the rough wooden beams forming the foreground structure, several residential buildings visible in the background, dense vegetation, and a piece of yellow agricultural or yard machinery located in the midground. **Activity or Event:** The image is static; there is no discernible human activity occurring, focusing instead on the structure and the surrounding environment. **Visible Text:** No clearly legible text is visible in the photograph. **Overall Mood:** The mood is raw and rustic, emphasizing the rough, exposed nature of the wooden structure and the dense, slightly cluttered residential setting."
gemma4:e2b/P3_hybrid,"Caption: A view through an overgrown wooden frame into a suburban residential backyard featuring houses, vegetation, and a piece of yellow machinery. Keywords: backyard, wooden frame, residential, vegetation, house, machinery, suburban, outdoor, fence"
gemma4:26b/P1_concise,A view of houses and greenery through a wooden pergola.
gemma4:26b/P2_struct,"This photo captures a quiet, outdoor residential setting, likely a backyard or a patio area, viewed from beneath a wooden pergola or overhead structure. **Place/Setting Type:** The setting is a residential area, specifically a backyard or courtyard, featuring several houses and buildings in the background and periphery. **Main Objects or Subjects:** * **Wooden Structure:** A prominent wooden pergola or patio cover with beams and a hanging light fixture dominates the upper part of the frame. * **Buildings:** Various residential structures are visible, including a building with grey siding and a window on the left, and another house in the background with a balcony where white laundry appears to be hanging. * **Machinery/Vehicle:** In the lower foreground, there is a piece of yellow machinery, which could be a small tractor or lawn mower. * **Nature and Fencing:** There are various green bushes and plants, along with a wire mesh fence on the left. * **Other items:** A light-colored tarp or cloth covers something in the bottom right corner. **Activity or Event:** There is no specific activity or event occurring; the scene is a static, candid shot of a backyard. **Visible Text:** There is no visible text in the image. **Overall Mood:** The overall mood is mundane, quiet, and peaceful, captured in the bright, clear light of a sunny day."
gemma4:26b/P3_hybrid,"Caption: A view of residential buildings and houses is framed by a wooden pergola structure. Keywords: pergola, wooden structure, house, balcony, greenery, light fixture, residential area, backyard"


...

gemma4:e2b/P1_concise,"A stylish woman poses casually in a colorful, narrow alley, showcasing a relaxed street style look."
gemma4:e2b/P2_struct,"This is a photograph of a young woman standing outdoors in a narrow, colorful urban setting. **Place/Setting Type:** The setting is a narrow street or alleyway flanked by buildings. The architecture suggests a residential or historic area, characterized by brightly painted facades. **Main Objects or Subjects:** The central subject is a young woman with long, light brown hair, standing facing the camera. She is wearing a light-colored (likely white) long-sleeved shirt or jacket and distressed denim shorts. She is holding a dark object (possibly a bag or clothing) near her right leg. The background features multiple buildings painted in contrasting colors, including light blue-gray, terracotta orange, and purple. Several potted plants and small trees line the street. Two bicycles are visible parked against the buildings on the right side. **Activity or Event:** The activity is a portrait or street style photo. The woman is posing while standing in the alleyway. **Visible Text:** Visible text is present on the buildings, appearing as numbers written on the walls (e.g., ""290"" and ""250"" are visible). **Overall Mood:** The mood is bright, casual, and relaxed, with warm lighting and vibrant colors contributing to a cheerful atmosphere."
gemma4:e2b/P3_hybrid,"Caption: A young woman poses in a vibrant, colorful urban alleyway lined with old buildings and greenery. Keywords: Woman, Street, Alley, Urban, Fashion, Bicycle, Plants, Clothing, Travel, Colorful"
gemma4:26b/P1_concise,"A stylish young woman poses in a scenic alleyway lined with vibrant, multi-colored buildings."
gemma4:26b/P2_struct,"This photograph features a young woman posing in a narrow, colorful alleyway. **Setting:** The scene is an outdoor urban street or alley lined with multi-colored buildings in shades of blue, orange, purple, and red. The ground appears dark and wet, suggesting recent rain. Lush green plants in pots and a bicycle parked against a wall are visible in the background. **Main Subject and Objects:** The central subject is a young woman with long, wavy brown hair. She is dressed in a white open cardigan, a white top, and light-blue distressed denim shorts. A pair of sunglasses hangs from her neckline, and she is holding a large, black wide-brimmed hat at the bottom of the frame. **Activity:** The woman is posing for a portrait, likely for a fashion or lifestyle photoshoot. **Visible Text:** The number ""250"" is visible on a small plaque on the purple building. **Overall Mood:** The mood of the image is calm, serene, and stylish, characterized by the soft lighting and the colorful, tranquil setting."
gemma4:26b/P3_hybrid,"Caption: A stylish young woman poses in a picturesque, colorful alleyway lined with vibrant buildings and lush greenery. Keywords: woman, street, alleyway, colorful buildings, travel, summer, plants, bicycle, architecture"


gemma4:e2b/P1_concise,"A wide, open vista of green plains and rolling hills stretches toward distant mountains under a bright blue sky."
gemma4:e2b/P2_struct,"This is an aerial photograph depicting a vast, open landscape under a bright, sunny sky. **Place/Setting Type:** The setting is a wide, open grassland or plain, bordered by distant, low-lying mountains or hills on the horizon. It appears to be a rural or semi-arid outdoor environment. **Main Objects or Subjects:** The foreground is dominated by a large expanse of vibrant green grass. A network of dirt paths or tracks crisscrosses the field, leading toward the background. In the middle ground, there are scattered small structures, possibly sheds or fences, and faint lines suggesting property boundaries or agricultural divisions. The background consists of undulating, rugged terrain and a clear blue sky filled with scattered white, fluffy cumulus clouds. **Activity or Event:** The photograph itself is an aerial view. The visible paths suggest routes for travel or access across the open land. There is no specific, discernible human activity being performed in the frame, but the setting implies an area used for traversing or viewing the landscape. **Visible Text:** No clearly discernible text or writing is visible within the main frame of the photograph. **Overall Mood:** The mood of the photograph is expansive, peaceful, and tranquil. The bright light, clear blue sky, and open scenery create a sense of openness and calm."
gemma4:e2b/P3_hybrid,"Caption: An expansive view of rolling green fields and distant mountains under a bright blue sky, showing an open, rural landscape. Keywords: landscape, fields, mountains, plains, sky, clouds, rural, open space, nature, scenery"
gemma4:26b/P1_concise,A vast green landscape featuring rolling hills and winding paths under a blue sky with white clouds.
gemma4:26b/P2_struct,"This is a wide-angle, high-angle shot of an expansive, lush green landscape, likely a steppe or grassland, featuring rolling hills and distant mountains. In the far distance, a small blue lake is visible nestled between the hills. The sky is a bright blue, filled with scattered, fluffy white cumulus clouds. In the foreground, a light-colored, curved gravel or dirt path bends through the green meadow, lined with several small, modern light poles. Winding through the vast landscape are various dirt tracks and paths. Scattered across the plains are several small white structures, possibly tents or small cabins, as well as a single, larger brown building to the right. The primary activity visible is movement along the paths, where several small figures, appearing to be people riding horses, can be seen traveling across the landscape. There is no visible text in the image. The overall mood of the photo is peaceful, serene, and vast, conveying a sense of quiet tranquility and immense open space."
gemma4:26b/P3_hybrid,"Caption: A vast green landscape with rolling hills and distant mountains stretches under a bright blue sky filled with white clouds. Keywords: grassland, mountains, sky, clouds, landscape, meadow, hills, nature"


gemma4:e2b/P1_concise,A soft-focus photograph captures a delicate abundance of white spring blossoms.
gemma4:e2b/P2_struct,"**Place/Setting Type:** Outdoors, focusing on dense foliage and blossoms. **Main Objects or Subjects:** Numerous clusters of small white and light pink flowers (blossoms) on branches, with soft green leaves visible in the background. **Activity or Event:** A close-up view of a flowering tree or bush in bloom. **Visible Text:** None. **Overall Mood:** Soft, delicate, gentle, and ethereal. The shallow depth of field creates a dreamy, romantic atmosphere."
gemma4:e2b/P3_hybrid,"Caption: A soft-focus close-up of branches covered in delicate white spring blossoms creates a dreamy, ethereal scene. Keywords: flowers, blossoms, spring, nature, white, bokeh, floral, spring bloom, branches"
gemma4:26b/P1_concise,"A soft-focus, close-up view of delicate white blossoms."
gemma4:26b/P2_struct,"This photo features a close-up, natural outdoor setting, such as a garden, park, or orchard. The main subjects are the branches of a tree densely covered in small, white and light pink blossoms, with a few green leaves interspersed throughout. The image uses a shallow depth of field, creating a soft, blurry effect in both the foreground and background. There is no visible text or specific activity occurring, though the scene depicts nature in bloom. The overall mood is peaceful, delicate, and serene."
gemma4:26b/P3_hybrid,"Caption: A soft-focus close-up capturing the delicate beauty of white blossoms in full bloom on tree branches. Keywords: blossoms, flowers, spring, nature, tree, cherry blossoms, bloom, garden, flora"


gemma4:e2b/P1_concise,A lush green lawn and vegetation frame a two-story residential building with visible windows and external air conditioning units.
gemma4:e2b/P2_struct,"**Place/Setting Type:** Residential outdoor area, likely a backyard or overgrown yard. **Main Objects or Subjects:** The image is dominated by a large expanse of lush green grass in the foreground. In the background is a two-story residential building with a light-colored facade. Visible features of the house include multiple windows, exterior railings, and several mounted external air conditioning units. Trees and dense foliage partially frame the scene. **Activity or Event:** There is no visible human activity or specific event taking place in the photograph. The focus is on the landscape and the structure. **Visible Text:** No legible text is visible in the photograph. **Overall Mood:** The mood is quiet, natural, and somewhat rustic, suggesting an area that is overgrown and peaceful."
gemma4:e2b/P3_hybrid,"Caption: An exterior shot of an older residential building surrounded by overgrown grass and vegetation. Keywords: house, exterior, grass, vegetation, building, window, yard, facade, residential"
gemma4:26b/P1_concise,A lush green garden sits in front of a multi-story brick building partially covered in climbing vines.
gemma4:26b/P2_struct,"This photograph depicts an outdoor residential or urban setting, featuring a multi-story building adjacent to a lush, green area. The main objects in the image include a building with light-colored tiled or brick walls, multiple windows, and several exterior air conditioning units mounted on the facade. Greenery is a prominent feature, with climbing vines growing on the building, a large evergreen tree framing the left side of the shot, and various bushes near the base of the structure. A small wooden object, which appears to be a bench or a small table, is partially visible among the bushes. The foreground is dominated by a large patch of green grass and low-lying plants. There are no people or specific activities occurring in the scene, and there is no legible text visible in the image. The overall mood of the photo is peaceful, quiet, and natural."
gemma4:26b/P3_hybrid,"Caption: A multi-story brick building is partially covered in green vines and sits behind a lush, grassy lawn. Keywords: building, lawn, grass, trees, brick wall, vines, garden, nature, apartment"


gemma4:e2b/P1_concise,"Silhouettes of several people stand along a beach, gazing at a warm sunset or sunrise over the water."
gemma4:e2b/P2_struct,"**Place/Setting Type:** A coastal landscape or beach setting, featuring water, landforms, and a wide expanse of sand or shore. **Main Objects or Subjects:** Silhouettes of several people in the foreground, the sun low on the horizon, distant landmasses (hills or mountains), and the body of water. **Activity or Event:** People are standing or walking along the shoreline, observing a sunset or sunrise over the water. **Visible Text:** There is no visible text in the image. **Overall Mood:** The mood is serene, dramatic, and tranquil, characterized by the warm, low light and the stark silhouettes of the figures against the bright horizon."
gemma4:e2b/P3_hybrid,"Caption: The photo captures a dramatic silhouette of people standing on a beach during a sunset or sunrise, with distant mountains on the horizon. Keywords: Sunset, Silhouette, Beach, Horizon, Landscape, Travelers, Seascape, Mountains, Ocean"
gemma4:26b/P1_concise,Silhouettes of people stand on a beach against a sunset with mountains in the background.
gemma4:26b/P2_struct,"This photograph depicts a coastal beach setting during sunset or sunrise. * **Place/Setting Type:** A beach or shoreline, featuring a vast expanse of dark sand in the foreground, the ocean in the midground, and silhouetted mountains or hills in the far distance. * **Main Objects or Subjects:** The primary subjects are the silhouetted figures of several people, the bright sun, the ocean, and the distant mountain range. * **Activity or Event:** A group of people is walking along the water's edge near the shoreline. * **Visible Text:** There is no visible text in the image. * **Overall Mood:** The mood is serene, peaceful, and tranquil."
gemma4:26b/P3_hybrid,"Caption: Silhouettes of people walking along a beach against the backdrop of a setting sun and distant mountains. Keywords: beach, sunset, mountains, ocean, silhouettes, people, coast, seascape, twilight, nature"


## §2 정성 평가 (사람 판단)

평균 길이(자): P1 e2b103/26b81 · P2 e2b843/26b819 · P3 e2b217/26b209.
- **P1_concise**: 1문장, 검색어휘 적음
- **P2_struct**: 슬롯 다 채움(장황). e2b는 마크다운 `**Place:**` 노출(노이즈)
- **P3_hybrid**: 서술+키워드 명시, 깔끔(검색 유리)
- **e2b vs 26b**: 26b 정확(반딧불이·pink jacket·차내 맥락), e2b는 어두운 장면 환각(반딧불이→나비) 경향
- 정확한 이미지 대조: 위 §2 표(썸네일+6캡션) 직접 확인 (로컬, ADR-0001 무관)

**→ 정량(§3) best: (e2b, P2_struct) + (e2b, P3_hybrid)** — 프롬프트 도출 집중, 모델 최종선택은 ⑤단계

## §3 정량 검색 recall + D19 판정

한국어 질의 → 캡션 임베딩 top-k recall. weak label = folder_top. 프롬프트 상대순위가 결론.

In [14]:
# §3 한국어 질의셋 (자동초안 + 사용자 보강). folder_top은 §0에서 NFC 정규화됨.
QUERIES = {
  "결혼식 사진": "wedding",
  "아이슬란드 여행": "2022_아이슬란드",
  "이탈리아 여행": "2019_이탈리아",
  "제주도에서 찍은 사진": "200620_23_제주",
  "방콕 여행": "2018방콕",
  "부산에서 찍은 사진": "181229_30_busan",
  "단양 여행": "201904_단양",
  "몽골 여행": "2018몽골",
  "개심사 절 사진": "20190505_개심사",
  "일산 호수공원": "20-04-05_일산호수공원",
  # TODO(사용자): 실제 물어볼 한국어 질의 추가/수정 (folder_top은 NFC)
}
bad = [v for v in QUERIES.values() if v not in set(m.folder_top)]
assert not bad, f"manifest에 없는 folder_top: {bad}"
assert 7 <= len(QUERIES) <= 20
print(len(QUERIES), "질의 · 정답 폴더 검증 OK")

10 질의 · 정답 폴더 검증 OK


In [15]:
# §3 평가 풀 ~400장 (정답폴더 폴더당 캡 + 디스트랙터)
gold_folders = set(QUERIES.values())
POS_CAP = 30   # 폴더당 최대 (이탈리아 583 등 캡)
pos = pd.concat([g.sample(min(POS_CAP, len(g)), random_state=SEED)
                 for _, g in m[m.folder_top.isin(gold_folders)].groupby("folder_top")])
n_neg = min(400 - len(pos), int((~m.folder_top.isin(gold_folders)).sum()))
neg = m[~m.folder_top.isin(gold_folders)].sample(n_neg, random_state=SEED)
pool = pd.concat([pos, neg]).drop_duplicates("thumb_path").reset_index(drop=True)
print("풀:", len(pool), "· 정답군:", len(pos), "· 디스트랙터:", len(neg), "· 폴더당캡:", POS_CAP)

풀: 400 · 정답군: 229 · 디스트랙터: 171 · 폴더당캡: 30


In [16]:
# §3 best 프롬프트 지정(정성 후 교체) + 풀 캡션 생성·캐시
BEST = [("gemma4:e2b", "P2_struct"), ("gemma4:e2b", "P3_hybrid")]   # §2 정성: P3 깔끔·26b 정확하나 느림 → e2b로 P2 vs P3
POOLCAP = CACHE / "pool_captions_03.parquet"
if POOLCAP.exists():
    pc = pd.read_parquet(POOLCAP); print("풀 캡션 캐시 로드:", len(pc))
else:
    recs, t0 = [], time.time()
    for _, row in pool.iterrows():
        for model, pid in BEST:
            recs.append({"thumb_path":row.thumb_path,"folder_top":row.folder_top,
                "model":model,"prompt_id":pid,"caption":caption(row.thumb_path,model,PROMPTS[pid])})
    pc = pd.DataFrame(recs); pc.to_parquet(POOLCAP)
    print(f"{len(pc)} 풀 캡션 · {time.time()-t0:.0f}s")

풀 캡션 캐시 로드: 800


In [17]:
# §3 임베딩 + recall@k 순수 함수 + assert
def embed(text):
    return np.asarray(ollama.embed(model=EMB_MODEL, input=text).embeddings[0], dtype=np.float32)

def recall_mrr(q_emb, pool_embs, pool_labels, gold, ks=(5,10)):
    sims = pool_embs @ q_emb / (np.linalg.norm(pool_embs,axis=1)*np.linalg.norm(q_emb)+1e-9)
    order = np.argsort(-sims)
    hits = [i for i,idx in enumerate(order) if pool_labels[idx]==gold]
    rr = 1.0/(hits[0]+1) if hits else 0.0
    return {f"recall@{k}": float(any(h < k for h in hits)) for k in ks} | {"mrr": rr}

_pe = np.array([[1,0],[0,1],[0.9,0.1]], dtype=np.float32)
_r = recall_mrr(np.array([1,0],dtype=np.float32), _pe, ["g","x","g"], "g")
assert _r["recall@5"]==1.0 and abs(_r["mrr"]-1.0)<1e-6, _r
print("recall_mrr assert OK")

recall_mrr assert OK


In [18]:
# §3 풀 임베딩 캐시 + 질의별 recall
POOLEMB = CACHE / "pool_embeddings_03.parquet"
if POOLEMB.exists():
    emb_df = pd.read_parquet(POOLEMB); print("임베딩 캐시 로드:", len(emb_df))
else:
    pc2 = pc.copy()
    pc2["emb"] = [embed(c).tolist() for c in pc2.caption]
    emb_df = pc2; emb_df.to_parquet(POOLEMB)
results = []
for (model,pid), g in emb_df.groupby(["model","prompt_id"]):
    P = np.array(g.emb.tolist(), dtype=np.float32); labels = g.folder_top.tolist()
    for q, gold in QUERIES.items():
        results.append({"model":model,"prompt_id":pid,"query":q,**recall_mrr(embed(q),P,labels,gold)})
res = pd.DataFrame(results)
print(len(res), "recall records")

임베딩 캐시 로드: 800


20 recall records


In [19]:
# §3 프롬프트 비교표 + D19 판정
print("== 프롬프트/모델별 평균 ==")
print(res.groupby(["model","prompt_id"])[["recall@5","recall@10","mrr"]].mean().round(3))
print("\n== 질의별 recall@10 ==")
print(res.pivot_table(index="query", columns=["model","prompt_id"], values="recall@10"))
mean_r10 = res.groupby(["model","prompt_id"])["recall@10"].mean().max()
print(f"\nD19 판정: 최고 recall@10 = {mean_r10:.2f} → "
      f"{'PASS(한국어<->영어캡션 매칭 작동)' if mean_r10>=0.5 else '관찰(보강 필요)'}")
print("※ 절대값은 weak label 한계로 참고. 프롬프트 상대순위가 결론.")

== 프롬프트/모델별 평균 ==
                      recall@5  recall@10    mrr
model      prompt_id                            
gemma4:e2b P2_struct       0.3        0.6  0.285
           P3_hybrid       0.5        0.7  0.340

== 질의별 recall@10 ==
model       gemma4:e2b          
prompt_id    P2_struct P3_hybrid
query                           
개심사 절 사진           1.0       1.0
결혼식 사진             1.0       1.0
단양 여행              1.0       0.0
몽골 여행              0.0       1.0
방콕 여행              1.0       1.0
부산에서 찍은 사진         0.0       1.0
아이슬란드 여행           1.0       1.0
이탈리아 여행            1.0       1.0
일산 호수공원            0.0       0.0
제주도에서 찍은 사진        0.0       0.0

D19 판정: 최고 recall@10 = 0.70 → PASS(한국어<->영어캡션 매칭 작동)
※ 절대값은 weak label 한계로 참고. 프롬프트 상대순위가 결론.


## §4 산출물

- `docs/01_eda_findings.md`에 "Vision 캡션 EDA" 섹션 — 선행파악·정성·recall·**best 프롬프트 전문**·모델비교·D19 판정
- wiki INGEST: `data-profile/eda-findings.md`, `models/model-decisions.md`
- ADR flag(사용자): D19/D20 신뢰도